# PLDAC - RL TABULAIRE

In [ ]:
try:
    from easypip import easyimport
except ModuleNotFoundError:
    from subprocess import run
    assert run(["pip", "install", "easypip"]).returncode == 0, "Could not install easypip"
    from easypip import easyimport

easyimport("swig")
easyimport("bbrl_utils").setup(maze_mdp=True)

import os
from pathlib import Path
from typing import List, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bbrl_gymnasium.envs.maze_mdp import MazeMDPEnv
from bbrl_utils.notebook import tqdm
from mazemdp.mdp import Mdp
from mazemdp.toolbox import egreedy, egreedy_loc, sample_categorical, softmax
from mazemdp import random_policy
import bbrl_gymnasium  # noqa: F401

In [ ]:
# matplotlib.use("TkAgg")

from functools import partial
!pip install optuna
import optuna
import yaml
import hydra
from omegaconf import OmegaConf, DictConfig

import seaborn as sns
sns.set_theme()
import logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

import time
from datetime import timedelta

In [ ]:
def renormalize(
    mdp: MazeMDPEnv,
    policy: np.array,
    state: np.array,
) -> None:
    """
    Renormalize the probability of actions so that the sum of probabilities over actions is always 1.
    We made sure in the calling function that the probabilities of action never get negative when they are decreased.
    :param mdp: the mdp the agent is working in (to get the number of actions)
    :param policy: the current policy, before normalization
    :param state: the state where the actions need to be renormalized
    :return: nothing
    """

    policy[state] = policy[state] / np.sum(policy[state])


In [ ]:
def perform_episode(
    env: MazeMDPEnv,
    policy: np.array,
    q: np.array,
    alpha_critic: float,
    alpha_actor: float,
    update_type: int,
    render: bool = True,
    stagnation_threshold: float = 0.1,
    max_no_improve: int = 10
) -> int:
    """
    Perform an episode on the given mdp with the given policy using Q-values.
    :param update_type: int indicating the chosen update rule (1, 2, 3, 4).
    :return: the number of steps before it stops
    """
    state, _ = env.reset(uniform=True)  # reinit env, get initial state
    terminated = False
    truncated = False
    steps = 0

    while not (terminated or truncated):
        action = sample_categorical(policy[state])  # draw action from the policy
        next_state, reward, terminated, truncated, _ = env.step(action)

        # max Q-value for the next state
        max_q_next = np.max(q[next_state])

        # next action according to the current policy (for SARSA)
        next_action = sample_categorical(policy[next_state])

        # TD error (delta) based on the selected update type
        if update_type == 1:
            # 1: SARSA update rule
            delta = reward + env.gamma * q[next_state, next_action] * (1 - terminated) - q[state, action]

        elif update_type == 2:
            # 2: Q-learning with max update rule
            delta = reward + env.gamma * max_q_next * (1 - terminated) - np.max(q[state])

        elif update_type == 3:
            # 3: MEAN SARSA update rule
            sum = np.sum(policy[state] * q[state])
            delta = reward + env.gamma * max_q_next * (1 - terminated) - sum
            
        elif update_type == 4:
            # 4: Q-learning update rule
            delta = reward + env.gamma * max_q_next * (1 - terminated) - q[state, action]

        else:
            raise ValueError("Invalid update type")

        # Update the Q-value with the delta
        q[state, action] += alpha_critic * delta

        # Update the policy with the new Q-values using the softmax function
        policy_temp = policy[state, action] + alpha_actor * delta
        policy[state, action] = max(policy_temp, 1e-8)

        renormalize(env, policy, state)
        assert np.isclose(np.sum(policy[state]), 1.0), "Policy is not properly normalized!"

        # Move to the next state
        state = next_state
        steps += 1

        value_norm = np.linalg.norm(q)

    return np.linalg.norm(q), steps


In [ ]:
def actor_critic_q(
    mdp: MazeMDPEnv,
    nb_episodes: int,
    alpha_critic: float,
    alpha_actor: float,
    update_type: int,
    render: bool = True,
) -> np.array:
    """
    Perform actor-critic training over a number of episodes with Q-values.
    :param update_type: int indicating the chosen update rule (1, 2, 3, 4).
    :return: a list of norm of Q values and a list of number of steps of episodes
    """
    # Initialize Q-values to zero
    q = np.zeros((mdp.nb_states, mdp.action_space.n))

    # Initialize uniform policy
    policy = np.ones((mdp.nb_states, mdp.action_space.n)) / mdp.action_space.n

    # Store the number of steps and norms
    nb_steps = []
    value_norms = []

    for ep in range(nb_episodes):
        print(f" Actor_critic_q function - looping Episode {ep + 1}/{nb_episodes}")
        value_norm, steps = perform_episode(mdp, policy, q, alpha_critic, alpha_actor, update_type, render)
        value_norms.append(value_norm)
        nb_steps.append(steps)

    return value_norms, nb_steps, policy, q

In [ ]:
def make_mdp(cfg):
    mdp = gym.make(cfg.mdp.name, kwargs={"width": cfg.mdp.width, "height": cfg.mdp.height, "ratio": cfg.mdp.ratio}, render_mode=cfg.mdp.render_mode)
    mdp = mdp.unwrapped
    return mdp

## Q-values Updates 
Expériences d’apprentissage pour tester 4 variantes de mise à jour des Q-values

In [ ]:
def run_experiment(cfg, nb_runs):
    """ Exécute l'algorithme Actor-Critic et retourne les résultats. """
    mdp = make_mdp(cfg)  # Créer un nouvel environnement avec les paramètres actuels
    update_names = ["SARSA", "Q-learning", "Mean SARSA", "Fourth Rule"]
    all_results = []
    summary = []

    for update_type in range(1, 5):
        all_steps = []

        for run in range(nb_runs):
            _, nb_steps = actor_critic_q(mdp, cfg.nb_episodes, cfg.alpha_critic, cfg.alpha_actor, update_type, cfg.render)
            all_steps.append(nb_steps)

        # Convertir en numpy arrays pour le calcul des statistiques
        all_steps = np.array(all_steps)
        mean_steps = np.mean(all_steps, axis=0)
        std_steps = np.std(all_steps, axis=0)

        all_results.append((update_names[update_type - 1], mean_steps, std_steps))
        summary.append([update_names[update_type - 1], np.mean(mean_steps), np.std(mean_steps)])

    return all_results, summary

In [ ]:
def plot_results(all_results, cfg):
    """ Affiche les courbes de performance en fonction des mises à jour et enregistre les plots. """

    # Créer le dossier "plots" si nécessaire
    if not os.path.exists('plots'):
        os.makedirs('plots')

    # Création du plot
    plt.figure(figsize=(12, 8))
    plt.title(f"Actor-Critic Performance - Comparison of Update Rules (gamma={cfg.mdp.gamma})")
    plt.xlabel("Episode")
    plt.ylabel("Number of Steps")
    plt.grid(True, linestyle="--", alpha=0.6)

    for name, mean_steps, std_steps in all_results:
        plt.plot(mean_steps, label=f"{name} - Mean Steps")
        plt.fill_between(range(cfg.nb_episodes), mean_steps - std_steps, mean_steps + std_steps, alpha=0.2)

    plt.legend(loc="upper right")

    # Générer un nom de fichier basé sur les paramètres
    plot_filename = f"actor_critic_comparison_gamma_{cfg.mdp.gamma}_alpha_critic_{cfg.alpha_critic}_alpha_actor_{cfg.alpha_actor}.pdf"
    plot_filepath = os.path.join('plots', plot_filename)

    # Sauvegarder le plot dans le dossier "plots"
    plt.savefig(plot_filepath)
    print(f"Plot saved to {plot_filepath}")

    # Afficher le plot
    plt.show()




In [ ]:
hyperparameter_tests = [
    {"alpha_critic": 0.8, "alpha_actor": 0.5},  # Critic fort
    {"alpha_critic": 0.5, "alpha_actor": 0.8},  # Actor fort
]

gammas = [0.9, 0.95, 0.99]  # Test sur différentes valeurs de gamma

N = 100  # Nombre de runs

for params in hyperparameter_tests:
    for gamma in gammas:
        print(f"\n\nTest avec alpha_critic={params['alpha_critic']}, alpha_actor={params['alpha_actor']} et gamma={gamma}")

        ac_params = {
            "save_curves": False,
            "save_heatmap": True,
            "mdp": {
                "name": "MazeMDP-v0",
                "width": 5,
                "height": 5,
                "ratio": 0.2,
                "render_mode": "rgb_array",
                "gamma": gamma,
            },
            "log_dir": "./tmp",
            "video_dir": "./tmp/videos",
            "nb_episodes": 100,
            "render": False,
            "nb_repeats": 5,
            "alpha_critic": params["alpha_critic"],
            "alpha_actor": params["alpha_actor"],
        }

        cfg = OmegaConf.create(ac_params)

        # Lancer le test et mesurer le temps d'exécution
        start_time = time.time()
        all_results, summary = run_experiment(cfg, N)
        end_time = time.time()

        elapsed_time = timedelta(seconds=(end_time - start_time))
        print(f"Temps d'exécution pour {N} runs avec gamma = {gamma} : {elapsed_time}\n")

        # Affichage des résultats
        summary_df = pd.DataFrame(summary, columns=["Update Rule", "Mean Steps", "Standard Deviation"])
        print("\nRésumé des résultats :\n", summary_df)

        plot_results(all_results, cfg)  # Tracer les résultats

## Performance par type de mise à jour 

In [ ]:
def plot_step_curve_per_rule(steps, update_rule_name, nb_episodes):
    plt.figure(figsize=(8, 5))
    plt.title(f"Steps per Episode - {update_rule_name}")
    plt.xlabel("Episode")
    plt.ylabel("Average Steps")
    plt.grid(True)

    steps = np.array(steps)
    mean_steps = steps.mean(axis=0)
    std_steps = steps.std(axis=0)

    plt.plot(mean_steps, label="Mean steps", color="orange")
    plt.fill_between(range(nb_episodes), mean_steps - std_steps, mean_steps + std_steps, alpha=0.2)
    plt.legend()
    plt.show()

In [ ]:
update_names = ["SARSA", "Q-learning (plus max)", "Mean SARSA", "Q-learning"]
static_results = []

nb_episodes = ac_params["nb_episodes"]
alpha_critic = ac_params["alpha_critic"]
alpha_actor = ac_params["alpha_actor"]
nb_runs = ac_params["nb_repeats"]

for update_type in range(1, 5):
    all_rewards = []
    all_steps = []
    for run in range(nb_runs):
        rewards, steps, _, _ = actor_critic_q(nb_episodes, alpha_critic, alpha_actor, update_type, render=False)
        all_rewards.append(rewards)
        all_steps.append(steps)

    static_results.append((update_names[update_type - 1], all_steps, all_rewards))


In [ ]:
for name, steps, _ in static_results:
    plot_step_curve_per_rule(steps, name, ac_params["nb_episodes"])

## Mesure de l'accord entre Actor et Critic

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Tuple, List

def compare_actor_critic_agreement(policy: np.ndarray, q: np.ndarray, show_table: bool = True, show_plot: bool = True) -> Tuple[float, List[int]]:
    """
    Compare actor's and critic's preferred actions for each state using np.where (handling ties),
    and visualize the agreement.

    :param policy: the learned policy (shape: nb_states x nb_actions)
    :param q: the learned Q-table (shape: nb_states x nb_actions)
    :param show_table: whether to display a pandas table of the agreement
    :param show_plot: whether to display a heatmap of the agreement
    :return: (agreement %, list of agreeing states)
    """
    nb_states = policy.shape[0]

    actor_actions = []
    critic_actions = []

    for s in range(nb_states):
        # Actor: choose actions with highest probability (policy)
        max_policy_val = np.max(policy[s])
        best_actor_actions = np.where(policy[s] == max_policy_val)[0]
        actor_actions.append(np.random.choice(best_actor_actions))

        # Critic: choose actions with highest Q-value
        max_q_val = np.max(q[s])
        best_critic_actions = np.where(q[s] == max_q_val)[0]
        critic_actions.append(np.random.choice(best_critic_actions))

    actor_actions = np.array(actor_actions)
    critic_actions = np.array(critic_actions)
    agreement = actor_actions == critic_actions

    df = pd.DataFrame({
        "State": range(nb_states),
        "Actor Action": actor_actions,
        "Critic Action": critic_actions,
        "Agreement": ["Y" if a else "N" for a in agreement]
    })

    agree_states = df[df["Agreement"] == "Y"]["State"].tolist()
    agree_percent = len(agree_states) / nb_states * 100

    if show_table:
        print("\n Agreement Table:")
        display(df)

    if show_plot:
        plt.figure(figsize=(12, 1))
        sns.heatmap([agreement.astype(int)], cmap="Greens", cbar=False, xticklabels=range(nb_states))
        plt.title(f"Actor-Critic Agreement Heatmap ({agree_percent:.2f}%)", fontsize=14)
        plt.xlabel("States")
        plt.yticks([])
        plt.show()

    return agree_percent, agree_states


In [ ]:
nb_episodes = 100
nb_runs = 5
alpha = 0.5

update_names = ["SARSA", "Q-learning (plus max)", "Mean SARSA", "Q-learning"]

for i in range(1, 5):
    all_steps = []
    print(f"\nUpdate type: {i} ({update_names[i-1]})")

    for run in range(nb_runs):
        _, steps, _, _ = actor_critic_q(
            nb_episodes=nb_episodes,
            alpha_critic=alpha,
            alpha_actor=alpha,
            update_type=i
        )
        all_steps.append(steps)

    # Affichage de la courbe
    plot_step_curve_per_rule(all_steps, update_names[i-1], nb_episodes)
